# Imports

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
np.set_printoptions(precision=4, floatmode='fixed', suppress=True)

mpl.rcParams['figure.figsize'] = (10,5)

sklearn.set_config(transform_output="pandas")

# Load Data

In [ ]:
df = pd.read_csv("datasets/dataset.csv", header=0, parse_dates=True, index_col=0)

In [ ]:
df.index

In [ ]:
df.info()

In [ ]:
df['type'].value_counts().sort_index()

In [ ]:
df['bedrooms'].value_counts().sort_index()

In [ ]:
df.head()

In [ ]:
df.tail()

# EDA

## Stationarity

In [ ]:
p_value = adfuller(df['price'], maxlag=50)[1]

In [ ]:
print(f"p-value = {p_value}")

## Dataviz

In [ ]:
plt.plot(df['price'], label=f"House Prices")
plt.title(f"House sales prices")
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
plt.scatter(df['bedrooms'], df['price'], color='skyblue')
plt.title(f"Prices vs number of bedrooms")
plt.tight_layout()
plt.show()

## Check Asimmetry

In [ ]:
sns.boxenplot(df, x='price')
plt.show()

In [ ]:
sns.histplot(df, x='price')
plt.show()

# Pre-Processing

> Log Transform

In [ ]:
df['price'] = np.log(df['price'])

In [ ]:
sns.boxenplot(df['price'])
plt.show()

In [ ]:
sns.histplot(df['price'])
plt.show()

> Remove column 'type'

In [ ]:
df.drop(labels='type', axis=1, inplace=True)

# Transform to Regular Time Series

> Grouping by time windows:\
> Aggregate the data at regular intervals within fixed windows.

In [ ]:
ts_df = df.resample('ME').mean()

In [ ]:
ts_df['price'].plot()
plt.show()

# Seasonal Decomposition

In [ ]:
sd = seasonal_decompose(ts_df['price'])
sd

In [ ]:
sd.plot()
plt.show()

> There is an upward trend and there is seasonality.

# Feature Engineering

> Year and month as explanatory variables

In [ ]:
ts_df['year'] = ts_df.index.year
ts_df['month'] = ts_df.index.month

> Round number of bedrooms

In [ ]:
ts_df['bedrooms'] = ts_df['bedrooms'].apply(lambda x: round(x))
ts_df.head()

# Split Data

> Keep time sequence

In [ ]:
n = int(len(ts_df) * 0.8)
n

In [ ]:
train = ts_df[:n]
test = ts_df[n:]

In [ ]:
X_train = train.drop(labels = 'price', axis=1)
y_train = train['price']

In [ ]:
X_train.head(10)

In [ ]:
X_test = test.drop(labels = 'price', axis=1)
y_test = test['price']

In [ ]:
X_test.head(10)

# Normalize

In [ ]:
scaler = StandardScaler()

In [ ]:
scaler.fit(X_train)

In [ ]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Linear Regression Model

In [ ]:
lr_model = LinearRegression()

In [ ]:
lr_model.fit(X_train_scaled, y_train)

In [ ]:
y_train_pred = lr_model.predict(X_train_scaled)

In [ ]:
train_mse = mean_squared_error(y_train, y_train_pred)

In [ ]:
plt.plot(train.index, np.exp(y_train), label=f"House prices (train)")
plt.plot(train.index, np.exp(y_train_pred), label=f"Estimated prices (train)")
plt.plot(train.index[0], np.exp(y_train[:1]), label = f"Train MSE: {train_mse:.4f}")
plt.legend()
plt.show()

In [ ]:
y_test_pred = lr_model.predict(X_test_scaled)

In [ ]:
test_mse = mean_squared_error(y_test, y_test_pred)

In [ ]:
plt.plot(test.index, np.exp(y_test), label=f"House prices (test)")
plt.plot(test.index, np.exp(y_test_pred), label=f"Estimated house prices (test)")
plt.plot(test.index[0], np.exp(y_test[:1]), label = f"Test MSE: {test_mse:.4f}")
plt.legend()
plt.show()

# Forecast

In [ ]:
X_test.tail()

In [ ]:
new_data = pd.DataFrame(
    data={
            'bedrooms': 3,
            'year': 2024,
            'month': 8
        },
    index=[pd.to_datetime("2024-08-31")]
) 

In [ ]:
new_data

In [ ]:
new_data_scaled = scaler.transform(new_data)
new_data_scaled

In [ ]:
lr_model.predict(new_data_scaled)

In [ ]:
prediction = np.exp(lr_model.predict(new_data_scaled )).item()
print(f"Prediction: {prediction:_.4f}")

In [ ]:
idx = 24

In [ ]:
plt.plot(y_test.index[-idx:], np.exp(y_test)[-idx:], label = f"Last {idx} records.")
plt.scatter(new_data.index, prediction, c = 'g', label=f"Predicted Price: {prediction:_.4f}")
plt.legend()
plt.show()

# End